[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-2-ml-dl-essentials/05-training-in-practice/micro-assignment/assignment.ipynb)

# Micro-assignment 2.5: Training in practice

Fill in each cell. Match the expected output in `README.md`. PyTorch only.

### Problem 1: A DataLoader batch

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader
X = torch.randn(100, 8)
y = torch.randint(0, 3, (100,))
# Wrap in a TensorDataset + DataLoader (batch_size=16), take the first batch, print X and y shapes.
# your code here


dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=16)

batch_X, batch_y = next(iter(loader))

print(f"batch X shape: {tuple(batch_X.shape)}")
print(f"batch y shape: {tuple(batch_y.shape)}")

batch X shape: (16, 8)
batch y shape: (16,)


### Problem 2: Train and evaluate

In [2]:
# Load load_digits, split off 25% test (random_state=0, stratify=y), scale on train,
# set torch.manual_seed(0), train Linear(64,32)-ReLU-Linear(32,10) with CrossEntropyLoss and
# Adam(lr=0.01) for 150 steps, then print whether the test accuracy exceeds 0.90.
# your code here

import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Load and split ---
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# --- Scale using train stats only (avoid leaking test info) ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# --- Convert to tensors ---
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# --- Model ---
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Train for 150 steps (full-batch, since no DataLoader specified) ---
for step in range(150):
    opt.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    opt.step()

# --- Evaluate ---
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_preds = test_logits.argmax(dim=1)
    test_acc = (test_preds == y_test_t).float().mean().item()

print(f"test accuracy above 0.90: {test_acc > 0.90}")

test accuracy above 0.90: True


### Problem 3: Save and reload give identical predictions

In [3]:
# Create a model, save its state_dict, load into a fresh model of the same shape,
# and print whether the two give identical predictions on a batch.
# your code here
import torch
import torch.nn as nn

# --- Create a model and a fresh batch to test on ---
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

batch = torch.randn(5, 64)

# --- Save its weights ---
torch.save(model.state_dict(), "model_weights.pt")

# --- Build a fresh model of the same shape and load the saved weights in ---
model2 = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)
model2.load_state_dict(torch.load("model_weights.pt"))

# --- Compare predictions on the same batch ---
model.eval()
model2.eval()
with torch.no_grad():
    pred1 = model(batch)
    pred2 = model2(batch)

identical = torch.equal(pred1, pred2)
print(f"identical: {identical}")

identical: True


### Problem 4: Read the overfitting onset

In [4]:
val_losses = [0.90, 0.61, 0.48, 0.52, 0.70]
# Print the epoch index with the lowest validation loss.
# your code here

best_epoch = val_losses.index(min(val_losses))

print(f"best epoch: {best_epoch}")

best epoch: 2


### Problem 5: A mismatch is a RuntimeError

In [5]:
# Trigger a shape mismatch with an incompatible matmul, catch it and print the error type,
# then fix the shapes and print the working result shape. Add the one-line note about device mismatch.
# your code here

import torch

# --- Trigger a shape mismatch on purpose ---
a = torch.randn(2, 3)
b = torch.randn(2, 3)  # incompatible: need a's columns to match b's rows

try:
    result = a @ b
except RuntimeError as e:
    print(f"error type: {type(e).__name__}")

# --- Fix it: reshape/transpose so inner dimensions line up ---
b_fixed = b.T  # now (3, 2), so (2,3) @ (3,2) works
result = a @ b_fixed

print(f"after fix, shape: {tuple(result.shape)}")

print("note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the same device")

error type: RuntimeError
after fix, shape: (2, 2)
note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the same device


### Problem 6: Reproducibility

In [6]:
# Write a helper that sets torch.manual_seed(seed) and returns float(torch.rand(1)).
# Call it twice with the same seed and print whether the values match.
# your code here

import torch

def get_random_value(seed):
    torch.manual_seed(seed)
    return float(torch.rand(1))

val1 = get_random_value(42)
val2 = get_random_value(42)

print(f"runs match: {val1 == val2}")

runs match: True
